# Skin Lesion Diagnosis Full Training Pipeline

## Environment Setup & Imports

In [2]:
import os
import sys
import torch

# Make project root importable from the notebook
PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import CFG, seed_everything, get_device
from src.dataset import (
    build_splits, get_dataloaders, compute_class_weights,
    SkinLesionDataset, get_val_transforms,
)
from src.models import build_model
from src.train import train_efficientnet, train_model
from src.evaluate import evaluate_model, build_comparison_table, plot_training_history
from src.gradcam import generate_gradcam_examples
from src.embeddings import run_embedding_analysis
from src.ablations import (
    study_augmentation, study_class_weights, study_backbone_freezing,
    build_ablation_summary,
)

print(f"PyTorch: {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")
print(f"Project root: {PROJECT_ROOT}")

PyTorch: 2.11.0+cu130  |  CUDA: True
Project root: /mnt/1080356580355308/skin_lesion_v2


## Configuration & Device

In [3]:
device      = get_device(CFG)
results_dir = CFG.paths.results

seed_everything(CFG.project.random_seed)
torch.set_float32_matmul_precision("high")

for path in [
    results_dir,
    CFG.paths.checkpoints,
    f"{results_dir}/gradcam",
    f"{results_dir}/embeddings",
    f"{results_dir}/ablations",
    f"{results_dir}/plots",
]:
    os.makedirs(path, exist_ok=True)

print(f"Results dir: {results_dir}")

[config] Device: cuda — NVIDIA GeForce RTX 5070 Ti Laptop GPU (12.3 GB VRAM)
[config] Seed fixed → 42
Results dir: /mnt/1080356580355308/skin_lesion_v2/results


## Section 3 — Model Registry

Central source of truth: `(model_key, display_name, checkpoint_subpath)`

Change `SELECTED_MODEL` to train only one model, or leave as `"all"` for the full pipeline.

In [5]:
MODEL_REGISTRY = [
    ("efficientnet", "EfficientNet-B3",  "efficientnet/final_best.pth"),
    ("resnet50",     "ResNet50",         "resnet50/best.pth"),
    ("vit",          "ViT-B/16 + LoRA",  "vit/best.pth"),
    ("simple_cnn",   "Simple CNN",       "simple_cnn/best.pth"),
]

SELECTED_MODEL = "all"   # options: "all", "efficientnet", "resnet50", "vit", "simple_cnn"
RUN_ABLATIONS  = True    # set False to skip ablation studies

print(f"Selected model : {SELECTED_MODEL}")
print(f"Run ablations  : {RUN_ABLATIONS}")

Selected model : all
Run ablations  : True


## Section 4 — Data Splits & Dataloaders

In [6]:
metadata_csv = f"{CFG.paths.data_raw}/HAM10000_metadata.csv"
train_df, val_df, test_df = build_splits(metadata_csv, CFG.paths.data_raw)

print(f"Train: {len(train_df):,}  Val: {len(val_df):,}  Test: {len(test_df):,}")

[dataset] Total valid images: 10015

  Train distribution:
    nv    :  4693  (66.9%)
    mel   :   779  (11.1%)
    bkl   :   769  (11.0%)
    bcc   :   360  (5.1%)
    akiec :   229  (3.3%)
    vasc  :    99  (1.4%)
    df    :    81  (1.2%)

  Val distribution:
    nv    :  1006  (67.0%)
    mel   :   167  (11.1%)
    bkl   :   165  (11.0%)
    bcc   :    77  (5.1%)
    akiec :    49  (3.3%)
    vasc  :    21  (1.4%)
    df    :    17  (1.1%)

  Test distribution:
    nv    :  1006  (66.9%)
    mel   :   167  (11.1%)
    bkl   :   165  (11.0%)
    bcc   :    77  (5.1%)
    akiec :    49  (3.3%)
    vasc  :    22  (1.5%)
    df    :    17  (1.1%)

[dataset] Split sizes — Train: 7010  Val: 1502  Test: 1503
Train: 7,010  Val: 1,502  Test: 1,503


In [7]:
class_weights = compute_class_weights(train_df)
train_loader, val_loader, test_loader = get_dataloaders(
    train_df, val_df, test_df, use_weighted_sampler=True,
)

print(f"Class weights: {class_weights}")

[dataset] Class weights (inverse frequency):
  nv    : n=4693  w=0.213
  mel   : n= 779  w=1.286
  bkl   : n= 769  w=1.302
  bcc   : n= 360  w=2.782
  akiec : n= 229  w=4.373
  vasc  : n=  99  w=10.115
  df    : n=  81  w=12.363
[dataset] Batches — Train: 219  Val: 47  Test: 47
Class weights: tensor([ 0.2134,  1.2855,  1.3022,  2.7817,  4.3731, 10.1154, 12.3633])


## Helper Functions

In [8]:
def ckpt_path(subpath: str) -> str:
    return os.path.join(CFG.paths.checkpoints, subpath)


def load_checkpoint(model: torch.nn.Module, subpath: str) -> torch.nn.Module:
    # weights_only=True avoids arbitrary code execution from pickle payloads
    model.load_state_dict(torch.load(ckpt_path(subpath), weights_only=True))
    return model


def train_one(model_key, model, train_loader, val_loader, class_weights, device):
    # EfficientNet uses a 3-stage fine-tuning schedule; all others use the generic loop
    if model_key == "efficientnet":
        return train_efficientnet(model, train_loader, val_loader, class_weights, device)
    return train_model(model, train_loader, val_loader, class_weights, device, model_name=model_key)

## Train Models

In [8]:
all_metrics   = {}
trained_models = {}

for model_key, display_name, ckpt_subpath in MODEL_REGISTRY:
    if SELECTED_MODEL not in ("all", model_key):
        continue

    print(f"\n{'=' * 25}\nTRAINING: {display_name}\n{'=' * }")

    model = build_model(model_key, device)

    if model_key == "efficientnet" and CFG.efficientnet.use_compile and torch.cuda.is_available():
        try:
            model = torch.compile(model)
            print("[main] torch.compile() applied")
        except Exception as exc:
            print(f"[main] torch.compile() skipped: {exc}")

    history = train_one(model_key, model, train_loader, val_loader, class_weights, device)
    plot_training_history(history, f"{results_dir}/plots", model_key)

    model = load_checkpoint(model, ckpt_subpath)
    eval_results = evaluate_model(model, test_loader, device,
                                  save_dir=results_dir, model_name=model_key)

    all_metrics[display_name]    = eval_results["metrics"]
    trained_models[model_key]    = model
    print(f"[{model_key}] Metrics: {eval_results['metrics']}")


TRAINING: EfficientNet-B3
[model] EfficientNet: backbone FROZEN — trainable params: 10,759
[model] Built 'efficientnet' | Total params: 10,706,991 | Device: cuda
[main] torch.compile() applied

STAGE 1 — Frozen backbone, training head only
[model] EfficientNet: backbone FROZEN — trainable params: 10,759


W0619 21:29:26.061000 10045 site-packages/torch/_inductor/utils.py:1731] [0/0] Not enough SMs to use max_autotune_gemm mode


  S1 Ep 01/5 | train_loss=1.0229 train_bacc=0.3462 | val_loss=1.7029 val_bacc=0.4531 val_f1=0.1591 | 85.9s
  ✓ New best val_balanced_acc = 0.4531 — checkpoint saved
  S1 Ep 02/5 | train_loss=0.7815 train_bacc=0.4438 | val_loss=1.3882 val_bacc=0.4959 val_f1=0.2258 | 30.9s
  ✓ New best val_balanced_acc = 0.4959 — checkpoint saved
  S1 Ep 03/5 | train_loss=0.7383 train_bacc=0.4933 | val_loss=1.4676 val_bacc=0.5303 val_f1=0.2629 | 31.4s
  ✓ New best val_balanced_acc = 0.5303 — checkpoint saved
  S1 Ep 04/5 | train_loss=0.7097 train_bacc=0.5040 | val_loss=1.4481 val_bacc=0.5529 val_f1=0.2627 | 30.6s
  ✓ New best val_balanced_acc = 0.5529 — checkpoint saved
  S1 Ep 05/5 | train_loss=0.6907 train_bacc=0.5042 | val_loss=1.3698 val_bacc=0.5641 val_f1=0.2743 | 30.6s
  ✓ New best val_balanced_acc = 0.5641 — checkpoint saved

STAGE 2 — Unfreezing last 2 backbone blocks
[model] EfficientNet: top 2 blocks UNFROZEN — trainable params: 8,516,837
  S2 Ep 01/15 | train_loss=0.6805 train_bacc=0.5158 | va

/home/nigus-haile/anaconda3/envs/lab_exercises/lib/python3.14/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
/home/nigus-haile/anaconda3/envs/lab_exercises/lib/python3.14/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
/home/nigus-haile/anaconda3/envs/lab_exercises/lib/python3.14/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
/home/nigus-haile/anaconda3/envs/lab_exercises/lib/python3.14/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()


  Ep 19/30 | train_loss=0.6977 train_bacc=0.4644 | val_loss=1.5729 val_bacc=0.4965 val_f1=0.2121 | 34.3s
  No improvement — patience 3/10


/home/nigus-haile/anaconda3/envs/lab_exercises/lib/python3.14/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
/home/nigus-haile/anaconda3/envs/lab_exercises/lib/python3.14/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
/home/nigus-haile/anaconda3/envs/lab_exercises/lib/python3.14/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
/home/nigus-haile/anaconda3/envs/lab_exercises/lib/python3.14/site-packages/albumentations/check_version.py:147: UserWarning: Error fetching version info <urlopen error [Errno -3] Temporary failure in name resolution>
  data = fetch_version_info()
/home/ni

  Ep 20/30 | train_loss=0.6979 train_bacc=0.4629 | val_loss=1.4432 val_bacc=0.5278 val_f1=0.2289 | 37.6s
  ✓ New best val_balanced_acc = 0.5278 — checkpoint saved
  Ep 21/30 | train_loss=0.6815 train_bacc=0.4707 | val_loss=1.5219 val_bacc=0.4969 val_f1=0.1824 | 40.2s
  No improvement — patience 1/10
  Ep 22/30 | train_loss=0.6742 train_bacc=0.4753 | val_loss=1.4266 val_bacc=0.5223 val_f1=0.1996 | 50.2s
  No improvement — patience 2/10
  Ep 23/30 | train_loss=0.6504 train_bacc=0.4896 | val_loss=1.4647 val_bacc=0.5075 val_f1=0.2371 | 51.2s
  No improvement — patience 3/10
  Ep 24/30 | train_loss=0.6418 train_bacc=0.4888 | val_loss=1.5201 val_bacc=0.4957 val_f1=0.2066 | 48.8s
  No improvement — patience 4/10
  Ep 25/30 | train_loss=0.6376 train_bacc=0.4961 | val_loss=1.4305 val_bacc=0.5358 val_f1=0.2541 | 57.6s
  ✓ New best val_balanced_acc = 0.5358 — checkpoint saved
  Ep 26/30 | train_loss=0.6484 train_bacc=0.4873 | val_loss=1.4597 val_bacc=0.5140 val_f1=0.2160 | 52.8s
  No improvement 

## Embedding Analysis (PCA + t-SNE)

In [9]:
for model_key, model in trained_models.items():
    print(f"[embeddings] {model_key}")
    run_embedding_analysis(model, test_loader, device,
                           save_dir=f"{results_dir}/embeddings",
                           model_name=model_key)

[embeddings] efficientnet
[embeddings] efficientnet: extracted 1503 features of dimension 1536
[embeddings] Warning: 1536 non-finite values replaced with 0. Check backbone for dead neurons or exploding activations.
[embeddings] PCA: top 50 components explain 74.9% variance
[embeddings] Running t-SNE (perplexity=30, max_iter=1000) ...
[embeddings] PCA scatter saved.
[embeddings] t-SNE scatter saved.
[embeddings] Interactive t-SNE HTML saved.
[embeddings] resnet50
[embeddings] resnet50: extracted 1503 features of dimension 2048
[embeddings] PCA: top 50 components explain 67.2% variance
[embeddings] Running t-SNE (perplexity=30, max_iter=1000) ...
[embeddings] PCA scatter saved.
[embeddings] t-SNE scatter saved.
[embeddings] Interactive t-SNE HTML saved.
[embeddings] vit
[embeddings] vit: extracted 1503 features of dimension 768
[embeddings] PCA: top 50 components explain 69.5% variance
[embeddings] Running t-SNE (perplexity=30, max_iter=1000) ...
[embeddings] PCA scatter saved.
[embeddin

## GradCAM Visualisations

EfficientNet only  requires a conv layer handle via `get_gradcam_layer()`.

In [10]:
if "efficientnet" in trained_models:
    model = trained_models["efficientnet"]
    # _orig_mod unwraps torch.compile's wrapper so GradCAM's .backward() works correctly
    gradcam_model = getattr(model, "_orig_mod", model)
    test_ds = SkinLesionDataset(test_df, transform=get_val_transforms(), return_path=True)
    generate_gradcam_examples(
        gradcam_model, gradcam_model.get_gradcam_layer(), test_ds, device,
        save_dir=f"{results_dir}/gradcam", model_name="efficientnet",
        n_correct=8, n_wrong=8,
    )
    print("[gradcam] Done.")
else:
    print("[gradcam] EfficientNet not trained — skipping.")

[gradcam] Saved 8 correct + 8 incorrect examples.
[gradcam] Done.


## Model Comparison Table

In [11]:
if len(all_metrics) > 1:
    build_comparison_table(all_metrics, results_dir)
    print("[comparison] Table saved.")
else:
    print("[comparison] Only one model trained — skipping comparison table.")


  MODEL COMPARISON
          Model Accuracy Balanced Acc Macro F1 ROC-AUC (macro)
EfficientNet-B3   0.3373       0.5154   0.2824          0.8468
       ResNet50   0.6094       0.7893   0.6415          0.9403
ViT-B/16 + LoRA   0.6567       0.7849   0.6793          0.9518
     Simple CNN   0.2023       0.4958   0.2296          0.8297
[comparison] Table saved.


## Ablation Studies

Three studies: augmentation strategies, class weighting, and backbone freezing depth.

In [9]:
if RUN_ABLATIONS and SELECTED_MODEL in ("all", "efficientnet"):
    print(f"\n{'=' * 60}\nRUNNING ABLATION STUDIES\n{'=' * 60}")
    ablation_dir = f"{results_dir}/ablations"
    abl_results  = {}
    abl_results.update(study_augmentation(train_df, val_df, device, ablation_dir))
    abl_results.update(study_class_weights(train_df, val_df, device, ablation_dir))
    abl_results.update(study_backbone_freezing(train_df, val_df, device, ablation_dir))
    build_ablation_summary(abl_results, ablation_dir)
    print("[ablations] Done.")
else:
    print("[ablations] Skipped.")


RUNNING ABLATION STUDIES

ABLATION STUDY 1 — Data Augmentation


[model] EfficientNet: backbone FROZEN — trainable params: 10,759
[model] EfficientNet: FULLY UNFROZEN — trainable params: 10,706,991
[dataset] Class weights (inverse frequency):
  nv    : n=4693  w=0.213
  mel   : n= 779  w=1.286
  bkl   : n= 769  w=1.302
  bcc   : n= 360  w=2.782
  akiec : n= 229  w=4.373
  vasc  : n=  99  w=10.115
  df    : n=  81  w=12.363

  [aug_off] aug=False | class_weights=True | freeze=False
    Ep 01/10  train_bacc=0.5287  val_bacc=0.6294
  ✓ New best val_balanced_acc = 0.6294 — checkpoint saved
    Ep 02/10  train_bacc=0.8060  val_bacc=0.7494
  ✓ New best val_balanced_acc = 0.7494 — checkpoint saved
    Ep 03/10  train_bacc=0.8949  val_bacc=0.7097
  No improvement — patience 1/5
    Ep 04/10  train_bacc=0.9288  val_bacc=0.7305
  No improvement — patience 2/5
    Ep 05/10  train_bacc=0.9482  val_bacc=0.7293
  No improvement — patience 3/5
    Ep 06/10  train_bacc=0.9588  val_bacc=0.7554
  ✓ New best val_balanced_acc = 0.7554 — checkpoint saved
    Ep 07/10  t

## Pipeline Complete

In [10]:
print(f"\n{'=' * 60}")
print("PIPELINE COMPLETE")
print(f"Results:   {results_dir}/")
print("Dashboard: streamlit run dashboard/app.py")
print("=" * 60)


PIPELINE COMPLETE
Results:   /mnt/1080356580355308/skin_lesion_v2/results/
Dashboard: streamlit run dashboard/app.py
